In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=4a54d49515baf7b41b1858f283c6007a34967a0e9bf84f3e5457b8ae0f64f7b2
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# ============================================================
# QUANTUM RANDOM BIT GENERATOR
# ============================================================

def quantum_random_bit():

    qc = QuantumCircuit(1,1)

    qc.h(0)

    qc.measure(0,0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    measured_bit = int(list(counts.keys())[0])

    return measured_bit

In [4]:
# ============================================================
# ALICE GENERATES RANDOM BITS AND BASES
# ============================================================

n = 20

alice_bits = []
alice_bases = []

for i in range(n):

    alice_bits.append(quantum_random_bit())
    alice_bases.append(quantum_random_bit())

print("Alice Bits:")
print(alice_bits)

print("\nAlice Bases:")
print(alice_bases)

Alice Bits:
[0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1]

Alice Bases:
[0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1]


In [5]:
# ============================================================
# ALICE ENCODES QUBITS
# ============================================================

qubits = []

for i in range(n):

    qc = QuantumCircuit(1,1)

    # Encode bit
    if alice_bits[i] == 1:
        qc.x(0)

    # Encode basis
    if alice_bases[i] == 1:
        qc.h(0)

    qubits.append(qc)

In [6]:
# ============================================================
# EVE INTERCEPTS QUBITS
# ============================================================

eve_bases = []
eve_results = []
eve_qubits = []

simulator = BasicSimulator()

for i in range(n):

    eve_basis = quantum_random_bit()

    eve_bases.append(eve_basis)

    qc = qubits[i].copy()

    # Eve measures in X basis
    if eve_basis == 1:
        qc.h(0)

    qc.measure(0,0)

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    eve_bit = int(list(counts.keys())[0])

    eve_results.append(eve_bit)

    # Eve resends qubit
    resend = QuantumCircuit(1,1)

    if eve_bit == 1:
        resend.x(0)

    if eve_basis == 1:
        resend.h(0)

    eve_qubits.append(resend)

print("Eve Bases:")
print(eve_bases)

print("\nEve Results:")
print(eve_results)

Eve Bases:
[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]

Eve Results:
[0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0]


In [7]:
# ============================================================
# BOB MEASURES QUBITS
# ============================================================

bob_bases = []
bob_results = []

for i in range(n):

    bob_basis = quantum_random_bit()

    bob_bases.append(bob_basis)

    qc = eve_qubits[i].copy()

    # Bob measures in X basis
    if bob_basis == 1:
        qc.h(0)

    qc.measure(0,0)

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    measured_bit = int(list(counts.keys())[0])

    bob_results.append(measured_bit)

print("Bob Bases:")
print(bob_bases)

print("\nBob Results:")
print(bob_results)

Bob Bases:
[0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1]

Bob Results:
[0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0]


In [8]:
# ============================================================
# CREATE SHARED KEYS
# ============================================================

shared_key_alice = []
shared_key_bob = []

for i in range(n):

    if alice_bases[i] == bob_bases[i]:

        shared_key_alice.append(alice_bits[i])
        shared_key_bob.append(bob_results[i])

print("Shared Key (Alice):")
print(shared_key_alice)

print("\nShared Key (Bob):")
print(shared_key_bob)

Shared Key (Alice):
[0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1]

Shared Key (Bob):
[0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0]


In [9]:
# ============================================================
# DETECT ATTACK
# ============================================================

errors = 0

for i in range(len(shared_key_alice)):

    if shared_key_alice[i] != shared_key_bob[i]:
        errors += 1

if len(shared_key_alice) > 0:
    error_rate = errors / len(shared_key_alice)
else:
    error_rate = 0

print("Error Rate:", error_rate)

if error_rate > 0.2:
    print("ATTACK DETECTED")
else:
    print("No attacker detected")

Error Rate: 0.16666666666666666
No attacker detected
